# Decision variants from the candidate dumps
Re-makes the answer-or-skip decision offline from `Decision_Dump_Run` dumps (every candidate of the 6 standard queries) and scores each variant against today's rule (most inliers + z, answer at ≥ 8 inliers).

Variants:
- **agree k**: also answer when ≥ k queries' own best candidates sit within r·σ of the chosen place (each with ≥ m inliers).
- **vote**: choose the place most queries support (any of their candidates within r·σ), then answer at ≥ 8 inliers or support ≥ 2.
- **mean position**: same answers, but the reported place is the inlier-weighted mean of the agreeing queries.

Env `DUMPS` = comma-separated dump folder names. Gains are paired bootstrap (95 %) in marks out of 60.

In [1]:
import os, sys, glob
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not ((ROOT / "src").is_dir() and (ROOT / "train/checks").is_dir()):
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from train.checks.vote_decision import load_photo, choose_max, query_support, choose_vote, mean_position, error_sigma
from train.checks.decision_gate import points_from_error, score_of_flags, paired_bootstrap_gain
RESULTS = ROOT / "tests/Test_Iteration_3/results/fresh_run"
DUMPS = os.environ.get("DUMPS", "dump_nudt_seed202_conf0.22,dump_official_conf0.22").split(",")
BAR = 8

In [2]:
def photo_variants(P):
    """One row per photo: error (sigma) and answer flag of every variant."""
    c, s = P["cand"], P["sigma"]
    row = dict(height=P["height"])
    k = choose_max(c)
    if k < 0:
        k = int(np.argmax(c["inliers"]))
    inl = c["inliers"][k]
    row["err_max"] = error_sigma(c["x"][k], c["y"][k], P["truth"], s)
    row["inl"] = inl
    row["ans_bar8"] = inl >= BAR
    for r in [0.5, 1.0, 2.0]:
        for m in [2, 3, 4]:
            a = query_support(c, c["x"][k], c["y"][k], r * s, m)
            row["agree_r%.1f_m%d" % (r, m)] = a
    # vote choice
    for r in [1.0, 2.0]:
        v, support = choose_vote(c, r * s, 3)
        if v < 0:
            row["err_vote_r%.1f" % r] = row["err_max"]; row["ans_vote_r%.1f" % r] = False
        else:
            row["err_vote_r%.1f" % r] = error_sigma(c["x"][v], c["y"][v], P["truth"], s)
            row["ans_vote_r%.1f" % r] = (c["inliers"][v] >= BAR) or (support >= 2)
    # averaged place around the max choice
    for r in [0.5, 1.0]:
        x, y = mean_position(c, k, r * s, 3)
        row["err_mean_r%.1f" % r] = error_sigma(x, y, P["truth"], s)
    return row

tables = {}
for name in DUMPS:
    files = sorted(glob.glob(str(RESULTS / name / "photo_*.npz")))
    rows = []
    for f in files:
        row = photo_variants(load_photo(f)); row["photo"] = int(Path(f).stem.split("_")[1]); rows.append(row)
    tables[name] = pd.DataFrame(rows)
    print(name, len(rows), "photos")

dump_nudt_seed202_conf0.22 200 photos


dump_official_conf0.22 379 photos


In [3]:
def report(t):
    base_pts = points_from_error(t.err_max.values); base_flags = t.ans_bar8.values
    out = []
    def add(label, pts, flags):
        s, w = score_of_flags(pts, flags)
        g = (np.where(flags, pts, 0) - np.where(base_flags, base_pts, 0)).astype(float)
        rng = np.random.default_rng(0)
        draws = [g[rng.integers(0, len(g), len(g))].mean() * 6 for _ in range(4000)]
        out.append(dict(variant=label, score=round(s, 2), wrong=w, answered=int(flags.sum()), gain=round(g.mean() * 6, 2),
                        lo=round(np.percentile(draws, 2.5), 2), hi=round(np.percentile(draws, 97.5), 2)))
    add("bar 8 (today)", base_pts, base_flags)
    add("always answer", base_pts, np.ones(len(t), bool))
    add("ceiling (answer iff points > 0)", base_pts, base_pts > 0)
    for r in [0.5, 1.0, 2.0]:
        for m in [2, 3, 4]:
            for k in [2, 3]:
                add("agree>=%d r=%.1f sigma m=%d" % (k, r, m), base_pts, base_flags | (t["agree_r%.1f_m%d" % (r, m)].values >= k))
    for r in [1.0, 2.0]:
        add("vote r=%.1f sigma" % r, points_from_error(t["err_vote_r%.1f" % r].values), t["ans_vote_r%.1f" % r].values)
    agree2 = base_flags | (t["agree_r1.0_m3"].values >= 2)
    for r in [0.5, 1.0]:
        add("agree>=2 + mean place r=%.1f" % r, points_from_error(t["err_mean_r%.1f" % r].values), agree2)
    return pd.DataFrame(out)

for name, t in tables.items():
    print("==", name, len(t), "photos")
    display(report(t))

== dump_nudt_seed202_conf0.22 200 photos


,variant,score,wrong,answered,gain,lo,hi
0,bar 8 (today),55.38,0,186,0.00,0.00,0.00
1,always answer,50.34,10,200,-5.04,-8.82,-1.44
2,ceiling (answer iff points > 0),56.34,0,189,0.96,0.18,2.04
3,agree>=2 r=0.5 sigma m=2,55.68,0,187,0.30,0.00,0.90
4,agree>=3 r=0.5 sigma m=2,55.68,0,187,0.30,0.00,0.90
5,agree>=2 r=0.5 sigma m=3,55.68,0,187,0.30,0.00,0.90
6,agree>=3 r=0.5 sigma m=3,55.68,0,187,0.30,0.00,0.90
7,agree>=2 r=0.5 sigma m=4,55.68,0,187,0.30,0.00,0.90
8,agree>=3 r=0.5 sigma m=4,55.38,0,186,0.00,0.00,0.00
9,agree>=2 r=1.0 sigma m=2,55.68,0,187,0.30,0.00,0.90


== dump_official_conf0.22 379 photos


,variant,score,wrong,answered,gain,lo,hi
0,bar 8 (today),49.01,0,316,0.00,0.00,0.00
1,always answer,54.17,9,379,5.16,2.31,8.04
2,ceiling (answer iff points > 0),57.02,0,369,8.01,6.05,10.07
3,agree>=2 r=0.5 sigma m=2,51.17,0,330,2.15,1.11,3.32
4,agree>=3 r=0.5 sigma m=2,49.49,0,319,0.47,0.00,1.11
5,agree>=2 r=0.5 sigma m=3,51.17,0,330,2.15,1.11,3.32
6,agree>=3 r=0.5 sigma m=3,49.49,0,319,0.47,0.00,1.11
7,agree>=2 r=0.5 sigma m=4,50.37,0,325,1.36,0.57,2.31
8,agree>=3 r=0.5 sigma m=4,49.33,0,318,0.32,0.00,0.79
9,agree>=2 r=1.0 sigma m=2,52.91,0,341,3.89,2.50,5.48


In [4]:
# precision: does the mean place shrink the error of answered photos? (photos answered under agree>=2, r = 1 sigma)
for name, t in tables.items():
    a = t[t.ans_bar8 | (t.agree_r1_0_m3 >= 2)] if "agree_r1_0_m3" in t else t[t.ans_bar8 | (t["agree_r1.0_m3"] >= 2)]
    a = a[a.err_max <= 8]
    print(name, "answered, within 8 sigma:", len(a))
    for col in ["err_max", "err_mean_r0.5", "err_mean_r1.0"]:
        print("  %-14s median %.3f  >1 sigma %.1f%%  >2 sigma %.1f%%" % (col, a[col].median(), 100 * (a[col] > 1).mean(), 100 * (a[col] > 2).mean()))
    band = pd.cut(a.height, [0, 150, 250, 400])
    print(a.assign(band=band).groupby("band", observed=True)[["err_max", "err_mean_r1.0"]].median().round(3).to_string())

dump_nudt_seed202_conf0.22 answered, within 8 sigma: 187
  err_max        median 0.405  >1 sigma 1.1%  >2 sigma 0.5%
  err_mean_r0.5  median 0.419  >1 sigma 1.1%  >2 sigma 0.5%
  err_mean_r1.0  median 0.405  >1 sigma 1.1%  >2 sigma 0.5%
            err_max  err_mean_r1.0
band                              
(150, 250]    4.769          4.769
(250, 400]    0.404          0.404
dump_official_conf0.22 answered, within 8 sigma: 341
  err_max        median 0.564  >1 sigma 5.0%  >2 sigma 0.0%
  err_mean_r0.5  median 0.545  >1 sigma 4.1%  >2 sigma 0.0%
  err_mean_r1.0  median 0.550  >1 sigma 4.7%  >2 sigma 0.0%
            err_max  err_mean_r1.0
band                              
(0, 150]      0.411          0.416
(150, 250]    0.533          0.544
(250, 400]    0.751          0.692
